# 任务 B：约束集校准（v11，条件式约束 + 无成本对照）

## 为什么是条件式

v10 的两档试图用「包裹型 + 行数预算」造出 0.8–0.95 的单条通过率，失败了：

| 约束 | DA | 说明 |
|---|---|---|
| `MAX_LINES_4` | 0.96 | CoT 恒为 0（trace 天生 9 行）→ **它是模式探测器，不是难度梯度** |
| `QUOTE` | 0.96 | 免费 |
| `SQUOTE` | 0.92 | 落在带内，但和 `QUOTE` 互斥，被贪心跳过 |
| `PAREN` | 0.56 | 太低 |
| `ANGLE` / `SPACED_ANSWER` | 0.00 | 模型完全不做 |

结论：**分类式约束只产出 0 或 1；定量式约束（行数）在 DA/CoT 之间天然分裂。**

v11 换一个思路：**条件式**。规则不再无条件要求某一行，而是
「**如果**某个条件成立，**才**写这一行；否则不许写」。这样：

- 通过率由**条件的基准率**控制（条件为真的题目比例），可以精确调到 0.8–0.95；
- 不依赖输出长度 → 不是模式探测器；
- 不同条件的失败落在**不同的题目子集**上 → 连乘能成立（这是 v9 缺的东西）。

## 6 条条件式

条件取自题面里**已知**的量，不需要查表：

| ID | 条件 | 要求的行 | 基准率（50 题） | 忽略规则时的 p |
|---|---|---|---|---|
| `S_AEUV` | 起点 ∈ {A,E,U,V} | `U1=N4` | 0.18 | 0.82 |
| `S_DFH` | 起点 ∈ {D,F,H} | `U2=P5` | 0.20 | 0.80 |
| `S_ZXYW` | 起点 ∈ {Z,X,Y,W} | `U3=Q6` | 0.16 | 0.84 |
| `A_AEUV` | **答案** ∈ {A,E,U,V} | `T7=R3` | 0.18 | 0.82 |
| `A_DFH` | **答案** ∈ {D,F,H} | `T8=K2` | 0.26 | 0.74 |
| `A_ZXYW` | **答案** ∈ {Z,X,Y,W} | `T9=L1` | 0.16 | 0.84 |

- `S_*` 用**起点**（题面直接给出，写在 header block 里没有时序问题）；
- `A_*` 用**模型自己的答案**（不是 gold），所以和答案正确率解耦——规则说的是「你输出的答案」。
  这也是它和 v6 取值型约束的区别：不需要读表，只需要看自己刚写下的那个字母。

**判分**：`(这一行恰好出现一次) == 条件是否成立`。条件不成立时写了也算错。
如果模型完全无视规则（永远不写），p 就等于 `1 - 基准率` ≈ 0.74–0.84；如果它永远写，p 等于基准率。
两种情况都在可测范围里。

## 6 条无成本对照

`C1`–`C6` = `T1=Q7` … `T6=H8`，v9/v10 实测 1.00/1.00。

## 预期

- 条件式 6 条的 `pass_DA` 落在 0.74–0.90；
- 阶梯 `k=1..5` 的 `joint_DA` 按 `Π p` 下降（因为条件子集互不重叠），k=5 约 0.3–0.4；
- 对照组 `C1`–`C6` 平坦在 1.00。

如果这两条都成立，**k 效应就是「独立条件失败率的连乘」**——这正是 IFEval 的机制，只是把它显式化成了条件规则。

## 流程

| 步骤 | 内容 | 产物 |
|---|---|---|
| 1 | 安装 vLLM（含 torchaudio 修复） | 环境 |
| 2 | 配置 + 读 50 题 | 固定 50 题 |
| 3 | 12 条约束（6 条件 + 6 常量） | `CONS` |
| 4 | 加载 4B | `llm` |
| 5 | 基准率表（零 GPU） | 预期 p |
| 6 | 冒烟：条件式 k=1/3/5 × 12 × 2 | 参考曲线 |
| 7 | 全量单约束：13 × 50 × 2 = 1300 条 | `calib_v11_cons.jsonl` |
| 8 | 两档各自建阶梯 | `calib_v11_cons_ladder.json` |
| 9 | 两条阶梯 × 50 × 2 = 800 条 | `calib_v11_ladder.jsonl` |


## 1. 安装 vLLM（Colab / Kaggle 专用）

本地已装好 vLLM 时跳过。Colab 安装后若提示 restart，重启会话，从第 2 格继续。

In [ ]:
import sys, os, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "pandas"], check=True)
    print("vLLM 安装完成。若提示 restart，请重启会话后从下一格开始。")
else:
    print("非 Colab/Kaggle 环境，跳过安装。")


# ---- 修 torchaudio / torch 的 CUDA 版本不匹配 ----
# 症状：from vllm import LLM 时，vllm -> transformers -> torchaudio 的 _check_cuda_version()
# 抛 "PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8"。
# 本项目是纯文本任务，用不到 torchaudio，直接卸掉最省事。
def drop_broken_torchaudio(verbose=True):
    if importlib.util.find_spec("torchaudio") is None:
        return False
    r = subprocess.run([sys.executable, "-c", "import torchaudio"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return False
    if verbose:
        tail = (r.stderr or "").strip().splitlines()[-1:] or ["(no stderr)"]
        print("[fix] torchaudio 导入失败：%s" % tail[0])
        print("[fix] 卸载 torchaudio（本项目不需要它），然后继续。")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    return True


_fixed = drop_broken_torchaudio()
if _fixed:
    print("[fix] 已卸载。若下面 import vllm 还报同样的错，请重启内核（Run -> Restart），再从第 2 格重跑。")


## 2. 配置

- 数据：优先读已上传的 `prompt_test_pointer_dataset.jsonl`，固定 50 题。
- 采样：`temperature=0.6`，`top_p=0.95`；DA 256 token，CoT 2048 token。
- 版本号 `v11-cond-k5`，产物 `calib_v11_*`。
- 全量：`F0 + 12 条` × 50 × 2 = **1300 条**；阶梯：两档各 5 档 × 50 × 2 = **800 条**。
- 没有 CoT 门槛；`DA_PASS_MIN = 0.70` 只挡明显坏掉的。


In [ ]:
import os, sys, json, re, time, random, math, gc, glob
from collections import Counter

# 必须在 import vllm 之前设置（AGENTS 第 3 节）
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

TIER = "B"
HF_ID = "Qwen/Qwen3-4B-Instruct-2507"
LOCAL_DIR = "models/Qwen3-4B-Instruct-2507"

PROMPT_VERSION = "v11-cond-k5"
SEED = 20261108
D_LIST = [5, 6, 7, 8, 9, 10]
N_PER_D = {5: 8, 6: 8, 7: 8, 8: 8, 9: 9, 10: 9}
N_PER_D_TOTAL = sum(N_PER_D.values())
MODES = ("DA", "CoT")
MAX_TOKENS = {"DA": 256, "CoT": 2048}
TEMPERATURE, TOP_P = 0.6, 0.95

# ---- k 轴 ----
K_MAX = 5
DA_PASS_MIN = 0.70
DA_BAND = (0.80, 0.95)

# ---- 两档 ----
COND_CONS = ["S_AEUV", "S_DFH", "S_ZXYW", "A_AEUV", "A_DFH", "A_ZXYW"]
CLASS_DA_FREE = ["C1", "C2", "C3", "C4", "C5", "C6"]

# 条件集合在下一节的 CONS 定义处给出（COND_SETS）

# ---- 阶段 ----
ORDER_STYLE = "Y"
SMOKE_PER_D = 2
SMOKE_KS = (1, 3, 5)
SMOKE_LADDER = ["S_AEUV", "A_AEUV", "S_DFH", "A_DFH", "S_ZXYW"]
RUN_LADDER = True
BAN_NONIDLE_BASELINE = "CoT"

CONST_LINES = {
    # 条件行（S_* 放在 header block，A_* 在答案附近）
    "S_AEUV": "U1=N4",
    "S_DFH": "U2=P5",
    "S_ZXYW": "U3=Q6",
    "A_AEUV": "T7=R3",
    "A_DFH": "T8=K2",
    "A_ZXYW": "T9=L1",
    # 无条件行
    "C1": "T1=Q7",
    "C2": "T2=K4",
    "C3": "T3=M9",
    "C4": "T4=Z2",
    "C5": "T5=W5",
    "C6": "T6=H8",
}

DATASET_NAME = "prompt_test_pointer_dataset.jsonl"

DATASET_CANDIDATES = [
    "Experiments/data_generation/" + DATASET_NAME,
    "data_generation/" + DATASET_NAME,
    DATASET_NAME,
    "/content/" + DATASET_NAME,
]
DATASET_CANDIDATES += sorted(glob.glob("/kaggle/input/**/" + DATASET_NAME, recursive=True))
DATASET_CANDIDATES += sorted(glob.glob("/content/**/" + DATASET_NAME, recursive=True))
DATASET_IN = next((p for p in DATASET_CANDIDATES if os.path.isfile(p)), None)

MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/cisc7404_calib"
elif IN_KAGGLE:
    OUT_DIR = "/kaggle/working"
else:
    OUT_DIR = "Experiments" if os.path.isdir("Experiments") else "."
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.isdir("Experiments/data_generation"):
    DATASET_OUT = "Experiments/data_generation/prompt_test_pointer_dataset.jsonl"
elif os.path.isdir("data_generation"):
    DATASET_OUT = "data_generation/prompt_test_pointer_dataset.jsonl"
else:
    DATASET_OUT = os.path.join(OUT_DIR, "prompt_test_pointer_dataset.jsonl")
os.makedirs(os.path.dirname(DATASET_OUT) or ".", exist_ok=True)

OUT_RAW = os.path.join(OUT_DIR, "calib_v11_cons.jsonl")
OUT_IO = os.path.join(OUT_DIR, "calib_v11_io.jsonl")
OUT_SUMMARY = os.path.join(OUT_DIR, "calib_v11_cons_summary.csv")
OUT_VERDICTS = os.path.join(OUT_DIR, "calib_v11_cons_verdicts.jsonl")
OUT_INDEP_F0 = os.path.join(OUT_DIR, "calib_v11_cons_independence_f0.csv")
OUT_INDEP_CROSS = os.path.join(OUT_DIR, "calib_v11_cons_independence_cross.csv")
OUT_LADDER = os.path.join(OUT_DIR, "calib_v11_cons_ladder.json")
OUT_LADDER_RAW = os.path.join(OUT_DIR, "calib_v11_ladder.jsonl")

print("MODEL:", HF_ID)
print("DATASET_IN:", DATASET_IN)
print("OUT_DIR:", OUT_DIR)
print("PROMPT_VERSION:", PROMPT_VERSION, "| K_MAX:", K_MAX, "| DA_PASS_MIN:", DA_PASS_MIN)
print("条件式:", COND_CONS)
print("无成本:", CLASS_DA_FREE)
print("全量:", 13 * N_PER_D_TOTAL * 2, "条 | 阶梯:", 2 * K_MAX * N_PER_D_TOTAL * 2, "条")
print("I/O 日志:", OUT_IO)


## 3. 生成指针追踪数据集

- **优先用已上传的 `prompt_test_pointer_dataset.jsonl`**（Kaggle input、仓库 `Experiments/data_generation/`、或 `/content/`）。
- 找不到才现场生成 50 题，并写到 `DATASET_OUT`。
- 已上传文件不会被覆盖，保证结果可复现。
- 每个 `(d, k, i)` 的题面唯一，`qid` 用 `d/k/i` 保证不冲突。
- 生成字段：
  - `start`：起点标签
  - `next`：走 1 步后的标签
  - `second`：走 2 步后的标签
  - `lastrow`：表格最后一行（`Z -> ?`）的右值
  - `firstrow`：表格第一行（`A -> ?`）的右值
  - `fanin_start` / `fanin_next`：有多少行的右值等于 `start` / `next`
  - `table_size`：表格行数（本数据固定 24）
  - `odd_d`：步数是否为奇数
  - `path`：完整路径，供人工审查

这些字段都只要求读题/扫表，不需要算最终答案，和 `answer_ok` 解耦。

In [ ]:
LABELS = list("ABCDEFGHJKLMNPQRSTUVWXYZ")  # 24 个，去掉 I / O


# ---------- 内置生成器：源文件缺失时使用，逻辑参考 synthesis_dataset.ipynb ----------
def _make_table(rng, n=len(LABELS), min_cycle_len=2):
    labels = LABELS[:n]
    cycle_len = rng.randint(min_cycle_len, n)
    cycle = rng.sample(labels, cycle_len)
    table = {}
    for i, node in enumerate(cycle):
        table[node] = cycle[(i + 1) % cycle_len]
    rest = [node for node in labels if node not in table]
    rng.shuffle(rest)
    attached = list(cycle)
    for node in rest:
        table[node] = rng.choice(attached)
        attached.append(node)
    return {node: table[node] for node in labels}


def _follow(table, start, steps):
    node = start
    for _ in range(steps):
        node = table[node]
    return node


def _table_text(table):
    return "\n".join("%s -> %s" % (node, table[node]) for node in LABELS)


def make_serial_question(rng, d):
    table = _make_table(rng, len(LABELS), d + 1)
    start = rng.choice(LABELS)
    ans = _follow(table, start, d)
    step_word = "step" if d == 1 else "steps"
    q = ("The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\n"
         "%s\n\n"
         "Start at %s. Follow the pointers for exactly %s %s.\n"
         "Question: which entry do you end at?") % (_table_text(table), start, d, step_word)
    return dict(q=q, ans=ans, d=d, W=d, k=0, i=0, table=table)


# ---------- 解析成约束判分字段 ----------
def parse_pointer_record(rec):
    q = rec["q"]
    m = re.search(r"Start at ([A-Z])\.", q)
    if not m:
        raise ValueError("找不到起点: " + q[:200])
    start = m.group(1)
    table = dict(rec["table"])
    if set(table) != set(LABELS) or len(table) != len(LABELS):
        raise ValueError("表不是 24 个标签")
    if start not in table:
        raise ValueError("起点不在表里")

    path = [start]
    node = start
    for _ in range(int(rec["d"])):
        node = table[node]
        path.append(node)
    ans = str(rec["ans"]).strip()
    if node != ans:
        raise ValueError("答案与表不一致: %s vs %s" % (node, ans))

    nxt = table[start]
    second = table[nxt]
    firstrow = table[LABELS[0]]
    lastrow = table[LABELS[-1]]
    return {
        "qid": "d%s_k%s_i%s" % (rec["d"], rec.get("k", 0), rec.get("i", 0)),
        "d": int(rec["d"]),
        "k": int(rec.get("k", 0)),
        "i": int(rec.get("i", 0)),
        "W": int(rec.get("W", rec["d"])),
        "q": q,
        "ans": ans,
        "table": table,
        "start": start,
        "next": nxt,
        "second": second,
        "lastrow": lastrow,
        "firstrow": firstrow,
        "fanin_start": sum(1 for v in table.values() if v == start),
        "fanin_next": sum(1 for v in table.values() if v == nxt),
        "table_size": len(table),
        "odd_d": bool(int(rec["d"]) % 2),
        "path": path,
    }


# ---------- 读取已上传数据集并重算派生字段；找不到才现场生成 ----------
def normalize_pointer_record(rec):
    """兼容旧 JSONL：只信 q/table/d/ans，其余派生字段全部重算。"""
    q = rec["q"]
    m = re.search(r"Start at ([A-Z])\.", q)
    if not m:
        raise ValueError("找不到起点: " + q[:200])
    start = m.group(1)
    table = {str(k): str(v) for k, v in rec["table"].items()}
    if set(table) != set(LABELS) or len(table) != len(LABELS):
        raise ValueError("表不是 24 个标签: " + str(rec.get("qid")))
    if start not in table:
        raise ValueError("起点不在表里: " + start)

    d = int(rec["d"])
    path = [start]
    node = start
    for _ in range(d):
        node = table[node]
        path.append(node)
    ans = str(rec["ans"]).strip()
    if node != ans:
        raise ValueError("答案与表不一致: %s vs %s" % (node, ans))

    nxt = table[start]
    return {
        "qid": str(rec.get("qid", "d%s_k%s_i%s" % (d, rec.get("k", 0), rec.get("i", 0)))),
        "d": d,
        "k": int(rec.get("k", 0)),
        "i": int(rec.get("i", 0)),
        "W": int(rec.get("W", d)),
        "q": q,
        "ans": ans,
        "table": table,
        "start": start,
        "next": nxt,
        "second": table[nxt],
        "lastrow": table[LABELS[-1]],
        "firstrow": table[LABELS[0]],
        "fanin_start": sum(1 for v in table.values() if v == start),
        "fanin_next": sum(1 for v in table.values() if v == nxt),
        "table_size": len(table),
        "odd_d": bool(d % 2),
        "path": path,
    }


if DATASET_IN:
    src_rows = [json.loads(l) for l in open(DATASET_IN, encoding="utf-8") if l.strip()]
    QUESTIONS = [normalize_pointer_record(r) for r in src_rows]
    print("使用已上传数据集:", DATASET_IN, "| 行数:", len(QUESTIONS))
    print("不覆盖该文件。要重新生成，把 DATASET_IN 置空。")
else:
    rng = random.Random(SEED)
    QUESTIONS = []
    for d in D_LIST:
        for j in range(N_PER_D[d]):
            rec = make_serial_question(rng, d)
            rec["i"] = j + 1
            QUESTIONS.append(parse_pointer_record(rec))
    with open(DATASET_OUT, "w", encoding="utf-8") as f:
        for q in QUESTIONS:
            f.write(json.dumps(q, ensure_ascii=False) + "\n")
    print("源数据集缺失，已现场生成:", DATASET_OUT, "| 行数:", len(QUESTIONS))

assert len(QUESTIONS) == N_PER_D_TOTAL, "题目数不是 %s: %s" % (N_PER_D_TOTAL, len(QUESTIONS))
assert len({q["qid"] for q in QUESTIONS}) == len(QUESTIONS)
assert len({q["q"] for q in QUESTIONS}) == len(QUESTIONS)

print("按 d:", dict(sorted(Counter(q["d"] for q in QUESTIONS).items())))
print("答案标签分布（前 8）:", Counter(q["ans"] for q in QUESTIONS).most_common(8))
print("\n第一题:", QUESTIONS[0]["qid"], "d=%s ans=%s" % (QUESTIONS[0]["d"], QUESTIONS[0]["ans"]))
print(QUESTIONS[0]["q"])
print({k: QUESTIONS[0][k] for k in ("start", "next", "second", "lastrow", "firstrow",
                                     "fanin_start", "fanin_next", "table_size", "odd_d")})

# 冒烟题集：每个 d 取前 SMOKE_PER_D 题（共 12 题）
SMOKE_QUESTIONS = [q for d in D_LIST for q in [x for x in QUESTIONS if x["d"] == d][:SMOKE_PER_D]]
assert len(SMOKE_QUESTIONS) == SMOKE_PER_D * len(D_LIST)
print("\n冒烟题集:", [q["qid"] for q in SMOKE_QUESTIONS])


## 4. 12 条约束：6 条条件式 + 6 条无条件

### 4.1 条件式（A 档，预期 p = 0.74–0.84）

| ID | 规则（英文原文见 `CONS`） | 条件 | 行 |
|---|---|---|---|
| `S_AEUV` | 起点 ∈ {A,E,U,V} 才写 | 题面已知 | `U1=N4` |
| `S_DFH` | 起点 ∈ {D,F,H} 才写 | 题面已知 | `U2=P5` |
| `S_ZXYW` | 起点 ∈ {Z,X,Y,W} 才写 | 题面已知 | `U3=Q6` |
| `A_AEUV` | 答案 ∈ {A,E,U,V} 才写 | 模型自己的答案 | `T7=R3` |
| `A_DFH` | 答案 ∈ {D,F,H} 才写 | 模型自己的答案 | `T8=K2` |
| `A_ZXYW` | 答案 ∈ {Z,X,Y,W} 才写 | 模型自己的答案 | `T9=L1` |

条件是**互不重叠**的集合，所以三条 `S_*` 的失败落在不同题目上，三条 `A_*` 也是。

### 4.2 为什么 `A_*` 用模型自己的答案

规则说的是「**你的**答案」。如果用 gold 答案判，就会把「答错了」算成「条件判断错」，和准确率强耦合——
record.md 第 2.6 节明确禁止内容轨进 k 轴。用模型自己的输出判，这条约束只测**条件判断 + 写入**，
和它答得对不对无关。

### 4.3 无条件对照（B 档）

`C1`–`C6` = `T1=Q7` … `T6=H8`，v9/v10 实测 1.00/1.00。

### 4.4 冲突表

这一轮的 12 条约束用的是**互不相同的 key**，没有逻辑互斥对（`conflict_list()` 为空）。
冲突表的机制保留：同 key 冲突、禁用字符撞必写字符这两条派生规则仍有自测（临时塞一个 `NO_EQUALS`）。

### 4.5 门槛

| 门槛 | 判据 |
|---|---|
| 结构 | `base_ok >= 0.90`、`placement >= 0.90`、`trunc < 5%`、`copy <= 1%`（不判草稿） |
| 非空转 | 要求类：F0(DA) 主动生成率 `<= 0.90` |
| 可用 | `pass_DA >= 0.70`，**没有 CoT 门槛** |
| 目标带 | `0.80–0.95` 只做标记 |


In [ ]:
MARK = "####ANSWER="

COND_SETS = {
    "AEUV": set("AEUV"),
    "DFH": set("DFH"),
    "ZXYW": set("ZXYW"),
}


def lines_of(t):
    return [l.strip() for l in str(t).strip().splitlines() if l.strip()]


def raw_lines(t):
    return str(t).splitlines()


def marker_lines(t):
    return [l for l in lines_of(t) if l.startswith(MARK)]


def final_raw(t):
    ls = marker_lines(t)
    if not ls:
        return None
    return ls[-1][len(MARK):].strip()


def marker_last_index(t):
    idx = [i for i, l in enumerate(lines_of(t)) if l.startswith(MARK)]
    return idx[-1] if idx else None


def is_trace_line(line):
    if re.fullmatch(r"[A-Za-z]\s*->\s*[A-Za-z]", line):
        return True
    if re.match(r"^Step\s*\d+\s*[:.]", line, re.I):
        return True
    if re.match(r"^(Entry reached|Final entry|Start at)\b", line, re.I):
        return True
    return False


def unwrap(s):
    if s is None:
        return None
    s = str(s).strip()
    for _ in range(2):
        if len(s) < 2:
            break
        if (s[0], s[-1]) in (("(", ")"), ("[", "]"), ('"', '"'), ("'", "'"), ("<", ">")):
            s = s[1:-1].strip()
        else:
            break
    return s


def final_label(t):
    s = unwrap(final_raw(t))
    if s is not None and re.fullmatch(r"[A-Za-z]", s):
        return s
    return None


# ---------------- 判定函数 ----------------
def _line_count(t, literal):
    return sum(1 for l in lines_of(t) if l == literal)


def _mk_cond(key, cond_set, source):
    """条件式：行的有无必须和条件一致。source ∈ {"start", "ans"}。"""
    literal = CONST_LINES[key]

    def _chk(t, r):
        if source == "start":
            value = str(r.get("start", "") or "").upper()
        else:
            lab = final_label(t)
            if lab is None:
                return False
            value = lab.upper()
        pred = value in COND_SETS[cond_set]
        return (_line_count(t, literal) == 1) == bool(pred)

    return _chk


def _mk_const(key):
    literal = CONST_LINES[key]

    def _chk(t, r):
        return _line_count(t, literal) == 1

    return _chk


CONS = {
    "F0": dict(track="base", cls="base", kind="base", keys=(), text=None, check=lambda t, r: True),
}

_COND_META = {
    "S_AEUV": ("start", "AEUV", "starting entry label"),
    "S_DFH": ("start", "DFH", "starting entry label"),
    "S_ZXYW": ("start", "ZXYW", "starting entry label"),
    "A_AEUV": ("ans", "AEUV", "final answer label"),
    "A_DFH": ("ans", "DFH", "final answer label"),
    "A_ZXYW": ("ans", "ZXYW", "final answer label"),
}

for _k, (_src, _set, _what) in _COND_META.items():
    _letters = ", ".join(sorted(COND_SETS[_set]))
    CONS[_k] = dict(
        track="cond", cls="cond", kind="require",
        keys=(CONST_LINES[_k].split("=")[0],), requires=("=",),
        check=_mk_cond(_k, _set, _src),
        text=("If the %s is one of %s, your response must contain the line %s, written exactly as shown. "
              "If the %s is not one of them, do not write that line. "
              "This conditional line must appear before the ####ANSWER= line." % (
                  _what, _letters, CONST_LINES[_k], _what)),
    )

for _k in CLASS_DA_FREE:
    CONS[_k] = dict(track="const", cls="da_free", kind="require",
                    keys=(CONST_LINES[_k].split("=")[0],), requires=("=",), check=_mk_const(_k),
                    text="Your response must contain the line %s, written exactly as shown. "
                         "Write it before the ####ANSWER= line." % CONST_LINES[_k])

CANDIDATES = COND_CONS + CLASS_DA_FREE
ALL_CANDIDATES = list(CANDIDATES)
assert len(CANDIDATES) == 12 and len(set(CANDIDATES)) == 12
assert all(CONS[c]["cls"] in ("cond", "da_free") for c in CANDIDATES)

LABEL_KEYS = tuple(sorted({k for c in CONS.values() for k in c.get("keys", ())}, key=len, reverse=True))
KEYLINE_RE = re.compile(r"^(" + "|".join(re.escape(k) for k in LABEL_KEYS) + r")\s*=")


# ---------------- 冲突表 ----------------
CONFLICT_PAIRS = {}


def conflict_reason(a, b):
    if a == b:
        return "同一条约束"
    pair = frozenset((a, b))
    if pair in CONFLICT_PAIRS:
        return CONFLICT_PAIRS[pair]
    if a not in CONS or b not in CONS:
        return None
    shared = set(CONS[a].get("keys", ())) & set(CONS[b].get("keys", ()))
    if shared:
        return "两条约束要求同一个 keyword 行: %s" % sorted(shared)
    fa, fb = set(CONS[a].get("forbids", ())), set(CONS[b].get("forbids", ()))
    ra, rb = set(CONS[a].get("requires", ())), set(CONS[b].get("requires", ()))
    if (fa & rb) or (fb & ra):
        return "一条禁用/要求了同一个字符: %s" % sorted((fa & rb) | (fb & ra))
    return None


def conflict_matrix(ids):
    m = pd.DataFrame("", index=ids, columns=ids, dtype=object)
    for a in ids:
        for b in ids:
            if conflict_reason(a, b):
                m.loc[a, b] = "X"
    return m


def conflict_list(ids=None):
    ids = list(ids or ALL_CANDIDATES)
    out = []
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            r = conflict_reason(a, b)
            if r:
                out.append({"a": a, "b": b, "reason": r})
    return out


def is_compatible(ids):
    return all(conflict_reason(a, b) is None for i, a in enumerate(ids) for b in ids[i + 1:])


# ---------------- 自测 ----------------
_r = dict(d=6, ans="R", start="S")
_t = ("U1=N4\nU2=P5\nU3=Q6\nT7=R3\nT8=K2\nT9=L1\nT1=Q7\n####ANSWER=R")

# 起点 S 不在任何集合里；答案是 R（不在任何集合里）→ 三个 S_* 三个 A_* 都应判「不该写」
assert not _mk_cond("S_AEUV", "AEUV", "start")(_t, _r)
assert not _mk_cond("A_AEUV", "AEUV", "ans")(_t, _r)
assert _mk_const("C1")(_t, _r)

_r2 = dict(d=6, ans="A", start="A")
assert _mk_cond("S_AEUV", "AEUV", "start")("U1=N4\n####ANSWER=A", _r2)
assert _mk_cond("A_AEUV", "AEUV", "ans")("T7=R3\n####ANSWER=A", _r2)
# 条件成立但不写 → 失败
assert not _mk_cond("S_AEUV", "AEUV", "start")("####ANSWER=A", _r2)
# 条件不成立却写了 → 失败
assert not _mk_cond("S_AEUV", "AEUV", "start")("U1=N4\n####ANSWER=R", dict(_r2, start="R"))
# 条件成立且写对 → 通过；写成别的也不对
assert not _mk_cond("A_DFH", "DFH", "ans")("T8=K2\n####ANSWER=A", _r2)
assert _mk_cond("A_DFH", "DFH", "ans")("T8=K2\n####ANSWER=D", dict(_r2, ans="D"))

# 冲突机制仍然自测（本轮池子无互斥对）
assert len(conflict_list()) == 0
assert is_compatible(CANDIDATES)
CONS["_TMP_BAN"] = dict(track="layout", cls="cond", kind="ban", keys=(), requires=(), forbids=("=",),
                        check=lambda t, r: "=" not in t, text="tmp")
assert conflict_reason("_TMP_BAN", "C1") is not None
del CONS["_TMP_BAN"]

print("判分 + 冲突表自测通过")
print("条件式:", COND_CONS)
print("无成本:", CLASS_DA_FREE)
print("行字面:", {k: CONST_LINES[k] for k in CANDIDATES})
print("互斥对:", len(conflict_list()))
print("冒烟临时阶梯:", SMOKE_LADDER)
assert is_compatible(SMOKE_LADDER)


## 5. Prompt 拼装

因为 `A_*` 的条件依赖**模型自己刚写下的答案**，它不可能在「header block 最先写」的骨架里完成，
所以 v11 把骨架退回到中性版本：

```
Answer with the final answer only: no calculations, no pointer steps, no intermediate values,
no explanation and no comment. Write only the lines that the rules below require, plus the final
answer line. Any other text makes the response invalid.

Rules:
R1. Output format. Write the lines required by the rules below first, in the order the rules are
    listed, and then the final answer line. The answer line is always the very last line ...
R2. If the starting entry label is one of A, E, U, V, your response must contain the line U1=N4 ...
```

v9 已经证明：只要字面是无意义代码，**有没有 header block 措辞都能把 `kw_after` 压到 0**
（v9 的 P 版没有 header block，`kw_after` 也是 0.00）。所以这里退回中性措辞是安全的。

每条条件式都自带两句：条件成立要写、不成立不许写；并说明「写在 ####ANSWER= 之前」。


In [ ]:
DA_PREAMBLE = ("Answer with the final answer only: no calculations, no pointer steps, no intermediate "
               "values, no explanation and no comment. Write only the lines that the rules below require, "
               "plus the final answer line. Any other text makes the response invalid.\n")

BASE_RULE = ("Output format. Write the lines required by the rules below first, in the order the rules are "
             "listed, and then write the final answer line. The answer line is always the very last line of "
             "the response: ####ANSWER= followed by the entry label as one letter, and nothing may be "
             "written after it. Do not repeat or restate these rules. Do not write any keyword line that "
             "these rules do not require.")

COT_PREAMBLE = ("Solve the problem step by step. Before the final answer line, explicitly write the entry "
                "reached after each pointer step, one per line. The keyword lines required by the rules "
                "below must come after those steps and before the final answer line.\n")


def preamble(mode, style=None):
    return COT_PREAMBLE if mode == "CoT" else DA_PREAMBLE


def rule_text(c, style=None):
    return CONS[c]["text"]


def build_cons(ids, style=None):
    rules = [BASE_RULE] + [rule_text(c) for c in ids if c != "F0"]
    parts = ["Rules:\n" + "\n".join("R%s. %s" % (j + 1, r) for j, r in enumerate(rules))]
    if not any(c != "F0" for c in ids):
        parts.append("None of the rules above requires a keyword line, so your response must be the final "
                     "answer line and nothing else.")
    return "\n\n".join(parts)


def build_messages(mode, ids, q, rec, style=None):
    return [
        {"role": "system", "content": preamble(mode, style) + build_cons(ids, style)},
        {"role": "user", "content": q["q"]},
    ]


def make_jobs(stage, questions, modes, cons_lists, style=None):
    jobs = []
    for mode in modes:
        for ids in cons_lists:
            cid = "+".join(ids)
            for q in questions:
                rec = {
                    "id": "%s|%s|%s|%s|%s|%s" % (PROMPT_VERSION, TIER, stage, mode, cid, q["qid"]),
                    "stage": stage,
                    "mode": mode,
                    "cons_id": cid,
                    "cons_ids": list(ids),
                    "k": len([c for c in ids if c != "F0"]),
                    "qid": q["qid"],
                    "d": q["d"],
                    "W": q["W"],
                    "q": q["q"],
                    "ans": q["ans"],
                    "start": q["start"],
                    "prompt_version": PROMPT_VERSION,
                    "tier": TIER,
                    "max_tokens": MAX_TOKENS[mode],
                }
                rec["messages"] = build_messages(mode, ids, q, rec, style)
                jobs.append(rec)
    return jobs


# ---- prompt 字面检查 ----
_KEY_ALT = "|".join(re.escape(k) for k in LABEL_KEYS)
_TEMPLATE_LINE = re.compile(r"(?m)^\s*(" + _KEY_ALT + r")\s*=")
_ANSWER_EXAMPLE = re.compile(r"(?m)^\s*####ANSWER=\s*[A-Za-z]")
for _ids in [["F0"]] + [[c] for c in ALL_CANDIDATES]:
    for _mode in MODES:
        _s = preamble(_mode) + build_cons(_ids)
        assert not _TEMPLATE_LINE.search(_s), ("行首模板", _mode, _ids)
        assert not _ANSWER_EXAMPLE.search(_s), ("答案样例", _mode, _ids)
        assert "<answer>" not in _s and "<label>" not in _s
        assert "Do not repeat" in _s
print("prompt 字面检查通过")

_jobs = make_jobs("selftest", QUESTIONS[:2], MODES, [["F0"], ["C1"], ["S_AEUV"]])
assert len(_jobs) == 2 * 3 * 2
assert len({j["id"] for j in _jobs}) == len(_jobs)
print("\n示例（S_AEUV / DA）:\n" + "-" * 90)
print([j for j in _jobs if j["cons_id"] == "S_AEUV" and j["mode"] == "DA"][0]["messages"][0]["content"])


## 6. 加载 vLLM（Qwen3-4B-Instruct-2507）

T4 / Turing 不支持 bf16，必须 `dtype="half"`；A100/L4 等可以走默认 bf16。
`Qwen3-4B-Instruct-2507` 是非 thinking 模型，`CHAT_KW` 保持空字典。

### 常见报错：torchaudio 与 torch 的 CUDA 版本不匹配

```
RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions.
PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8.
```

原因：`from vllm import LLM` 会连带 import `transformers.processing_utils`，后者无条件 import `torchaudio`，
而 `torchaudio/_extension` 在 import 时就会校验 CUDA 版本，不匹配直接抛异常。

处理：**卸掉 torchaudio**。本项目是纯文本任务，vLLM 不需要它。第 1 格和第 6 格都带了自动检查，
会先试 import、失败就 `pip uninstall torchaudio`。

如果已经踩过这个错、且当前内核里 `transformers` 被部分 import 过，**必须先 Restart 内核**再从头跑，
否则残留的半初始化模块还会继续报错。


In [ ]:
import torch

# 再兜一次：如果第 1 格被跳过（环境里 vLLM 已经装好），这里也能修
import sys as _sys, subprocess as _sp, importlib.util as _ilu
if _ilu.find_spec("torchaudio") is not None:
    _r = _sp.run([_sys.executable, "-c", "import torchaudio"], capture_output=True, text=True)
    if _r.returncode != 0:
        print("[fix] torchaudio 不可用，卸载后继续")
        _sp.run([_sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)

from vllm import LLM, SamplingParams

if not torch.cuda.is_available():
    raise RuntimeError("没有 CUDA。请到 Kaggle / Colab 的 GPU 运行时执行这一格。")

model_path = LOCAL_DIR if os.path.isdir(LOCAL_DIR) else HF_ID
cap = torch.cuda.get_device_capability(0)
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30

kw = dict(
    model=model_path,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    max_num_seqs=32,
)
if cap[0] < 8:
    kw["dtype"] = "half"

print("LLM kwargs:", kw)
print("GPU: %.1f GiB, capability %s" % (mem_gib, cap))
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
llm = LLM(**kw)

# Qwen3-4B-Instruct-2507 是非 thinking 模型：不要传 enable_thinking
CHAT_KW = {}
SP = {m: SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS[m]) for m in MAX_TOKENS}
print("SamplingParams:", {m: (TEMPERATURE, TOP_P, MAX_TOKENS[m]) for m in MAX_TOKENS})


## 7. 生成器

- 结果追加写 JSONL，按 `id` 断点续跑。
- I/O 日志 `calib_v10_io.jsonl`：一行一条 `{id, stage, mode, cons_id, k, qid, d, system, user, output, finish_reason, n_tokens, base_ok, answer_ok, placement_ok, kw_after_answer, strict_cons_ok, per_cons}`。
- `da_extra` / `da_trace` / `da_other` 仍然落盘，但只作诊断。


In [ ]:
SCORE_FIELDS = (
    "base_ok", "answer_ok", "placement_ok", "kw_after_answer", "cons_only", "cons_ok",
    "strict_cons_ok", "per_cons", "per_cons_all", "da_extra", "da_trace", "da_other",
    "cot_trace_ok", "copy_flag", "n_lines", "n_commas", "n_digits", "has_colon", "has_blank",
)


def load_done(path):
    if not os.path.exists(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(l)["id"] for l in f if l.strip()}


def run_jobs(jobs, path=OUT_RAW, io_path=OUT_IO):
    done = load_done(path)
    todo = [j for j in jobs if j["id"] not in done]
    print("总计 %s，已完成 %s，待跑 %s" % (len(jobs), len(jobs) - len(todo), len(todo)))
    for mode in MODES:
        part = [j for j in todo if j["mode"] == mode]
        batch = 64 if mode == "DA" else 32
        for s in range(0, len(part), batch):
            chunk = part[s:s + batch]
            t0 = time.time()
            outs = llm.chat([j["messages"] for j in chunk], SP[mode], use_tqdm=False, **CHAT_KW)
            dt = time.time() - t0
            with open(path, "a", encoding="utf-8") as f, open(io_path, "a", encoding="utf-8") as fio:
                for j, o in zip(chunk, outs):
                    c = o.outputs[0]
                    rec = dict(j)
                    msgs = rec.pop("messages", None)
                    rec.update(
                        text=c.text,
                        prompt_tokens=len(o.prompt_token_ids),
                        n_tokens=len(c.token_ids),
                        finish_reason=c.finish_reason,
                        trunc=(c.finish_reason == "length"),
                        batch_sec=round(dt, 2),
                        batch_size=len(chunk),
                    )
                    sc = score(rec)
                    rec.update({k: sc[k] for k in SCORE_FIELDS})
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    if msgs:
                        fio.write(json.dumps({
                            "id": rec["id"], "stage": rec["stage"], "mode": rec["mode"],
                            "cons_id": rec["cons_id"], "k": rec["k"], "qid": rec["qid"], "d": rec["d"],
                            "system": msgs[0]["content"], "user": msgs[1]["content"],
                            "output": c.text, "finish_reason": c.finish_reason,
                            "n_tokens": rec["n_tokens"],
                            "base_ok": sc["base_ok"], "answer_ok": sc["answer_ok"],
                            "placement_ok": sc["placement_ok"],
                            "kw_after_answer": sc["kw_after_answer"],
                            "strict_cons_ok": sc["strict_cons_ok"],
                            "per_cons": sc["per_cons"],
                        }, ensure_ascii=False) + "\n")
            print("[%s] %s/%s  batch %.1fs" % (mode, s + len(chunk), len(part), dt))


def _required_keys(rec):
    keys = set()
    for c in rec.get("cons_ids", []) or []:
        if c in CONS:
            keys.update(CONS[c].get("keys", ()))
    return keys


def kw_after_answer(t, rec):
    idx = marker_last_index(t)
    if idx is None:
        return False
    keys = _required_keys(rec)
    for i, line in enumerate(lines_of(t)):
        if i <= idx:
            continue
        km = KEYLINE_RE.match(line)
        if km and km.group(1) in keys:
            return True
    return False


def score(rec):
    t = rec["text"]
    mode = rec["mode"]
    is_da = mode == "DA"
    lines = lines_of(t)
    label = final_label(t)
    mlines = marker_lines(t)

    base_ok = len(mlines) == 1 and label is not None
    placement_ok = marker_last_index(t) == len(lines) - 1
    after = kw_after_answer(t, rec)
    answer_ok = label is not None and label.upper() == str(rec["ans"]).upper()

    per_all = {c: bool(CONS[c]["check"](t, rec)) for c in ALL_CANDIDATES}
    ids = [c for c in rec.get("cons_ids", ["F0"]) if c != "F0"]
    per = {c: per_all[c] for c in ids}
    cons_only = all(per.values()) if per else True

    allowed = set()
    for c in ids:
        allowed.update(CONS[c].get("keys", ()))

    other = []
    for line in lines:
        if line.startswith(MARK):
            continue
        km = KEYLINE_RE.match(line)
        if km and km.group(1) in allowed:
            continue
        other.append(line)

    da_trace = bool(is_da and any(is_trace_line(l) for l in other))
    da_other = bool(is_da and any(not is_trace_line(l) for l in other))
    da_extra = bool(is_da and other)

    trace = [l for l in lines if not l.startswith(MARK) and not KEYLINE_RE.match(l)]
    cot_trace_ok = (not is_da) and len(trace) >= max(1, rec["d"] // 2)
    copy_flag = len(mlines) >= 3 or re.search(r"<(?:answer|label|number|value)>", t, re.I) is not None

    return {
        "base_ok": bool(base_ok),
        "answer_ok": bool(answer_ok),
        "placement_ok": bool(placement_ok),
        "kw_after_answer": bool(after),
        "cons_only": bool(cons_only),
        "cons_ok": bool(base_ok and cons_only),
        "strict_cons_ok": bool(base_ok and placement_ok and cons_only),
        "per_cons": per,
        "per_cons_all": per_all,
        "da_extra": da_extra,
        "da_trace": da_trace,
        "da_other": da_other,
        "cot_trace_ok": bool(cot_trace_ok),
        "copy_flag": bool(copy_flag),
        "n_lines": len(lines),
        "n_commas": t.count(","),
        "n_digits": len(re.findall(r"\d", t)),
        "has_colon": ":" in t,
        "has_blank": re.search(r"\n[ \t]*\n", str(t).strip()) is not None,
    }


def load_results(path=OUT_RAW, stage=None):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    if stage:
        rows = [r for r in rows if r.get("stage") == stage]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    sc = [score(r) for r in rows]
    df = df.drop(columns=[c for c in SCORE_FIELDS if c in df.columns])
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(sc)], axis=1)


## 8. 基准率表（零 GPU）

条件式的通过率主要由**条件的基准率**决定：如果模型完全无视规则（永远不写那一行），
`p = 1 - 基准率`；如果它永远写，`p = 基准率`。所以先算清楚 50 题里每个条件为真的比例。

这一格不跑模型，只读数据。


In [ ]:
rows = []
for _c in COND_CONS:
    _src, _set, _what = _COND_META[_c]
    _vals = [str(q["start" if _src == "start" else "ans"]).upper() for q in QUESTIONS]
    _base = sum(1 for v in _vals if v in COND_SETS[_set]) / len(_vals)
    rows.append({
        "cons": _c,
        "source": _src,
        "set": "".join(sorted(COND_SETS[_set])),
        "base_rate": round(_base, 3),
        "p_if_ignore": round(1 - _base, 3),
        "p_if_always": round(_base, 3),
        "line": CONST_LINES[_c],
    })
base_tab = pd.DataFrame(rows).set_index("cons")
display(base_tab)

print("\n如果模型完全无视规则，6 条的预期 joint_DA:")
_prod = 1.0
for _c, _row in base_tab.iterrows():
    _prod *= (1 - _row["base_rate"])
    print("  %-8s p=%.2f -> 连乘 %.3f" % (_c, 1 - _row["base_rate"], _prod))


## 9. 冒烟：条件式临时阶梯 k=1 / 3 / 5

`S_AEUV → A_AEUV → S_DFH → A_DFH → S_ZXYW` 的前缀，12 题 × 3 档 × 2 模式 = 72 条。

这三条 `S_*` 和两条 `A_*` 条件互不重叠，所以 joint 应该按连乘掉。只看趋势，不阻断。


In [ ]:
SMOKE_LISTS = [SMOKE_LADDER[:k] for k in SMOKE_KS]
smoke_jobs = make_jobs("smoke", SMOKE_QUESTIONS, MODES, SMOKE_LISTS)
run_jobs(smoke_jobs)
sm = load_results(stage="smoke")
assert not sm.empty, "冒烟没有结果"

display(sm.groupby(["k", "mode"]).agg(
    n=("id", "size"),
    cons_pass=("strict_cons_ok", "mean"),
    base=("base_ok", "mean"),
    place=("placement_ok", "mean"),
    kw_after=("kw_after_answer", "mean"),
    n_lines=("n_lines", "median"),
).round(3))

jrow = sm.groupby(["k", "mode"])["strict_cons_ok"].mean().unstack("mode").reindex(columns=["DA", "CoT"])
print("\n条件式 joint 曲线:")
display(jrow.round(3))
for k in jrow.index:
    print("k=%s: DA=%.3f CoT=%.3f" % (k, jrow.loc[k, "DA"], jrow.loc[k, "CoT"]))
SMOKE_OK = True


## 10. 正式单约束校准

`F0` + 12 条，各 50 题，DA/CoT，共 **1300 条**。不阻断。


In [ ]:
CONS_LISTS = [["F0"]] + [[c] for c in CANDIDATES]
jobs = make_jobs("cons_calib", QUESTIONS, MODES, CONS_LISTS)
print("jobs:", len(jobs))
run_jobs(jobs)

cal = load_results(stage="cons_calib")
print("已完成行数:", len(cal))
print(cal.groupby(["cons_id", "mode"]).size().unstack("mode").to_string())

## 11. 汇总与可行性判定

- 没有 CoT 门槛；`feasible` = 结构 + 非空转 + `pass_DA >= 0.70`。
- `0.80–0.95` 只是标记（`in_band`）。
- 分档报：`cond`（条件式）与 `da_free`（无条件）。


In [ ]:
BASE_MIN = 0.90
TRUNC_MAX = 0.05
COPY_MAX = 0.01
DA_OTHER_MAX = 0.20
ANSWER_MIN = 0.70       # 只告警


def _pass_all(d, c):
    return bool(d.get(c, False)) if isinstance(d, dict) else False


def wilson_ci(p, n, z=1.96):
    if n <= 0:
        return (float("nan"), float("nan"))
    p = min(max(float(p), 0.0), 1.0)
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (centre - half, centre + half)


def newcombe_gap_ci(p1, n1, p2, n2, z=1.96):
    if n1 <= 0 or n2 <= 0:
        return (float("nan"), float("nan"))
    l1, u1 = wilson_ci(p1, n1, z)
    l2, u2 = wilson_ci(p2, n2, z)
    d = p1 - p2
    lo = d - math.sqrt(max(0.0, (p1 - l1) ** 2 + (u2 - p2) ** 2))
    hi = d + math.sqrt(max(0.0, (u1 - p1) ** 2 + (p2 - l2) ** 2))
    return (lo, hi)


def gap_slope_wls(gd):
    xs = gd.index.astype(float).values
    pda = gd["DA"].values.astype(float)
    pcot = gd["CoT"].values.astype(float)
    n = np.maximum(gd["n"].values.astype(float), 1.0)
    var = np.maximum(pda * (1 - pda) / n + pcot * (1 - pcot) / n, 1e-6)
    w = 1.0 / var
    ys = pda - pcot
    sw = w.sum()
    xbar = float((w * xs).sum() / sw)
    ybar = float((w * ys).sum() / sw)
    sxx = float((w * (xs - xbar) ** 2).sum())
    if sxx <= 1e-12:
        return float("nan"), float("nan")
    return float((w * (xs - xbar) * (ys - ybar)).sum() / sxx), float(math.sqrt(1.0 / sxx))


def summarize(cal):
    if cal.empty:
        raise RuntimeError("没有校准结果。先运行第 10 格。")
    cal = cal.copy()
    cal["track"] = cal["cons_id"].map(lambda c: CONS.get(c, {}).get("track", "?"))
    cal["cls"] = cal["cons_id"].map(lambda c: CONS.get(c, {}).get("cls", "?"))

    g = cal.groupby(["cons_id", "mode"])
    summ = g.agg(
        n=("id", "size"),
        cons_pass=("strict_cons_ok", "mean"),
        cons_value=("cons_ok", "mean"),
        cons_only=("cons_only", "mean"),
        base_ok=("base_ok", "mean"),
        placement=("placement_ok", "mean"),
        answer=("answer_ok", "mean"),
        trunc=("trunc", "mean"),
        da_extra=("da_extra", "mean"),
        da_other=("da_other", "mean"),
        kw_after=("kw_after_answer", "mean"),
        copy=("copy_flag", "mean"),
        n_tokens=("n_tokens", "mean"),
        n_lines=("n_lines", "median"),
    )
    summ["cons_pass_stop"] = cal[~cal["trunc"].astype(bool)].groupby(
        ["cons_id", "mode"])["strict_cons_ok"].mean()
    summ = summ.round(3)

    pv = summ["cons_pass_stop"].unstack("mode").reindex(columns=["DA", "CoT"])
    bo = summ["base_ok"].unstack("mode").reindex(columns=["DA", "CoT"])
    pl = summ["placement"].unstack("mode").reindex(columns=["DA", "CoT"])
    tr = summ["trunc"].unstack("mode").reindex(columns=["DA", "CoT"])
    cp = summ["copy"].unstack("mode").reindex(columns=["DA", "CoT"])
    de = summ["da_other"].unstack("mode").reindex(columns=["DA", "CoT"])
    ka = summ["kw_after"].unstack("mode").reindex(columns=["DA", "CoT"])
    an = summ["answer"].unstack("mode").reindex(columns=["DA", "CoT"])
    nn = summ["n"].unstack("mode").reindex(columns=["DA", "CoT"])

    f0 = cal[cal["cons_id"] == "F0"]
    f0_pass = {}
    for mode in MODES:
        sub = f0[f0["mode"] == mode]
        f0_pass[mode] = {
            c: (float(sub["per_cons_all"].map(lambda d: _pass_all(d, c)).mean()) if len(sub) else float("nan"))
            for c in ALL_CANDIDATES
        }

    lo_band, hi_band = DA_BAND
    rows = []
    for c in [x for x in ALL_CANDIDATES if x in pv.index]:
        kind = CONS[c]["kind"]
        baseline = BAN_NONIDLE_BASELINE if kind == "ban" else "DA"
        fp = f0_pass[baseline].get(c, float("nan"))
        if kind == "ban":
            non_idle_value = 1.0 - fp
            non_idle = bool(pd.notna(fp) and non_idle_value >= 0.15)
        else:
            non_idle_value = fp
            non_idle = bool(pd.notna(fp) and non_idle_value <= 0.90)

        pass_da = float(pv.loc[c, "DA"]) if pd.notna(pv.loc[c, "DA"]) else float("nan")
        pass_cot = float(pv.loc[c, "CoT"]) if pd.notna(pv.loc[c, "CoT"]) else float("nan")
        n_da = float(nn.loc[c, "DA"]) if pd.notna(nn.loc[c, "DA"]) else 0.0
        n_cot = float(nn.loc[c, "CoT"]) if pd.notna(nn.loc[c, "CoT"]) else 0.0
        gap = pass_da - pass_cot
        gap_lo, gap_hi = newcombe_gap_ci(pass_da, n_da, pass_cot, n_cot)

        sub = cal[(cal["cons_id"] == c) & (~cal["trunc"].astype(bool))]
        per_d = []
        for dd, s3 in sub.groupby("d"):
            da3 = s3[s3["mode"] == "DA"]
            cot3 = s3[s3["mode"] == "CoT"]
            if len(da3) and len(cot3):
                per_d.append((float(dd), float(da3["strict_cons_ok"].mean()),
                              float(cot3["strict_cons_ok"].mean()), float(len(da3))))
        if len(per_d) >= 2:
            gd = pd.DataFrame(per_d, columns=["d", "DA", "CoT", "n"]).set_index("d")
            gap_slope, gap_se = gap_slope_wls(gd)
        else:
            gap_slope, gap_se = float("nan"), float("nan")

        base_min = float(bo.loc[c].min(skipna=True))
        placement_min = float(pl.loc[c].min(skipna=True))
        trunc_max = float(tr.loc[c].max(skipna=True))
        copy_max = float(cp.loc[c].max(skipna=True))
        da_other_max = float(de.loc[c].max(skipna=True))
        answer_da = float(an.loc[c, "DA"]) if pd.notna(an.loc[c, "DA"]) else float("nan")
        structure_ok = (base_min >= BASE_MIN and placement_min >= BASE_MIN and
                        trunc_max < TRUNC_MAX and copy_max <= COPY_MAX)
        feasible = bool(structure_ok and non_idle and pd.notna(pass_da) and pass_da >= DA_PASS_MIN)

        failed = []
        if not structure_ok:
            failed.append("结构(base=%.3f,place=%.3f,trunc=%.3f,copy=%.3f)" % (
                base_min, placement_min, trunc_max, copy_max))
        if not non_idle:
            failed.append("非空转(F0.%s=%.3f)" % (baseline, fp))
        if not (pd.notna(pass_da) and pass_da >= DA_PASS_MIN):
            failed.append("DA通过率=%.3f" % pass_da)

        rows.append({
            "cons_id": c,
            "cls": CONS[c]["cls"],
            "track": CONS[c]["track"],
            "kind": kind,
            "f0_pass_DA": f0_pass["DA"].get(c, float("nan")),
            "non_idle_baseline": baseline,
            "non_idle": non_idle,
            "pass_DA": pass_da,
            "pass_CoT": pass_cot,
            "in_band": bool(pd.notna(pass_da) and lo_band <= pass_da <= hi_band),
            "gap": gap,
            "gap_ci_low": gap_lo,
            "gap_ci_high": gap_hi,
            "gap_slope": gap_slope,
            "base_min": base_min,
            "placement_min": placement_min,
            "trunc_max": trunc_max,
            "kw_after_max": float(ka.loc[c].max(skipna=True)),
            "da_other_max": da_other_max,
            "answer_DA": answer_da,
            "structure_ok": bool(structure_ok),
            "feasible": feasible,
            "reason": "; ".join(failed) if failed else "可行",
        })

    verdict = pd.DataFrame(rows).set_index("cons_id")
    print("逐约束汇总:")
    display(summ)
    print("\n分档可行性:")
    display(verdict.round(3))
    for cls in ("cond", "da_free"):
        sub = verdict[verdict["cls"] == cls]
        if len(sub):
            print("\n[%s] n=%d, feasible=%d, 平均 pass_DA=%.3f, 落在 0.80-0.95 的=%d" % (
                cls, len(sub), int(sub["feasible"].sum()), sub["pass_DA"].mean(),
                int(sub["in_band"].sum())))
    return summ, verdict


## 12. 分档建阶梯

两档各自按 `pass_DA` 从高到低贪心取 5 条（冲突则跳过），得到两条独立的 k=1..5 阶梯：

- **A 档阶梯**：如果有单条失败率，`joint_DA(k)` 应该按 `p_1·p_2···p_k` 下降 —— 这就是 IFEval 的连乘。
- **B 档阶梯**：p=1.00，`joint` 应该恒为 1 —— 对照组。

两条曲线的差就是「单条失败率」的贡献，也就是 k 效应的全部来源（在这个模型上）。


In [ ]:
from itertools import combinations


def conditional_matrix(df, candidates):
    p = pd.DataFrame(index=candidates, columns=candidates, dtype=float)
    n = pd.DataFrame(index=candidates, columns=candidates, dtype=int)
    if df is None or len(df) == 0:
        return p, n
    for a in candidates:
        pass_a = df["per_cons_all"].map(lambda d: _pass_all(d, a))
        n_a = int(pass_a.sum())
        for b in candidates:
            if a == b:
                continue
            n.loc[a, b] = n_a
            if n_a >= 10:
                viol_b = ~df["per_cons_all"].map(lambda d: _pass_all(d, b))
                p.loc[a, b] = float((pass_a & viol_b).sum() / n_a)
    return p, n


def conditional_cross(cal, candidates):
    p = pd.DataFrame(index=candidates, columns=candidates, dtype=float)
    n = pd.DataFrame(index=candidates, columns=candidates, dtype=int)
    for a in candidates:
        sub = cal[cal["cons_id"] == a]
        if sub.empty:
            continue
        pass_a = sub["per_cons_all"].map(lambda d: _pass_all(d, a))
        n_a = int(pass_a.sum())
        for b in candidates:
            if a == b:
                continue
            n.loc[a, b] = n_a
            if n_a >= 10:
                viol_b = ~sub["per_cons_all"].map(lambda d: _pass_all(d, b))
                p.loc[a, b] = float((pass_a & viol_b).sum() / n_a)
    return p, n


def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    return str(o)


def build_ladder(verdict, pool_ids, k_max=K_MAX):
    """按 pass_DA 从高到低贪心取 k_max 条，跳过冲突。"""
    pool = [c for c in pool_ids if c in verdict.index and bool(verdict.loc[c, "feasible"])]
    pool.sort(key=lambda c: (-float(verdict.loc[c, "pass_DA"]), -float(verdict.loc[c, "pass_CoT"])))
    chosen, skipped = [], []
    for c in pool:
        if len(chosen) >= k_max:
            break
        hit = [(x, conflict_reason(c, x)) for x in chosen if conflict_reason(c, x)]
        if hit:
            skipped.append({"cons_id": c, "blocked_by": hit[0][0], "reason": hit[0][1]})
            continue
        chosen.append(c)
    return chosen, skipped, pool


def project_joint(ladder, verdict):
    rows = []
    for k, ids in enumerate(ladder, 1):
        pda = float(np.prod([float(verdict.loc[c, "pass_DA"]) for c in ids]))
        pcot = float(np.prod([float(verdict.loc[c, "pass_CoT"]) for c in ids]))
        rows.append({"k": k, "constraints": "+".join(ids), "joint_DA": pda,
                     "joint_CoT": pcot, "joint_gap": pda - pcot})
    return pd.DataFrame(rows).set_index("k").round(3)


def finalize(cal):
    summ, verdict = summarize(cal)
    all_run = [c for c in ALL_CANDIDATES if c in verdict.index]
    f0 = cal[cal["cons_id"] == "F0"]
    indep_f0, indep_f0_n = conditional_matrix(f0, all_run)
    indep_cross, indep_cross_n = conditional_cross(cal, all_run)

    ladders, joints, chosen_all, skipped_all = {}, {}, {}, {}
    for cls, ids in (("cond", COND_CONS), ("da_free", CLASS_DA_FREE)):
        chosen, skipped, pool = build_ladder(verdict, ids, K_MAX)
        chosen_all[cls] = chosen
        skipped_all[cls] = skipped
        ladders[cls] = [chosen[:k] for k in range(1, len(chosen) + 1)]
        joints[cls] = project_joint(ladders[cls], verdict) if ladders[cls] else pd.DataFrame()

    summ.to_csv(OUT_SUMMARY, encoding="utf-8")
    indep_f0.to_csv(OUT_INDEP_F0, encoding="utf-8")
    indep_cross.to_csv(OUT_INDEP_CROSS, encoding="utf-8")

    with open(OUT_VERDICTS, "w", encoding="utf-8") as f:
        for c in ALL_CANDIDATES:
            if c in verdict.index:
                rec = verdict.loc[c].to_dict()
                rec["cons_id"] = c
                rec["status"] = "feasible" if bool(verdict.loc[c, "feasible"]) else "rejected"
                rec["in_ladder"] = any(c in chosen_all[k] for k in chosen_all)
            else:
                rec = {"cons_id": c, "status": "not_run", "reason": "未运行"}
            f.write(json.dumps(rec, ensure_ascii=False, default=_json_default) + "\n")

    payload = {
        "prompt_version": PROMPT_VERSION,
        "model": HF_ID,
        "order_style": ORDER_STYLE,
        "k_max": K_MAX,
        "n_questions": len(QUESTIONS),
        "gates": {"base_min": BASE_MIN, "da_pass_min": DA_PASS_MIN,
                  "trunc_max": TRUNC_MAX, "copy_max": COPY_MAX},
        "da_band": list(DA_BAND),
        "conflicts": conflict_list(),
        "classes": {"cond": COND_CONS, "da_free": CLASS_DA_FREE},
        "ladders": {
            cls: [
                {"k": int(k),
                 "constraints": [c for c in str(row["constraints"]).split("+") if c],
                 "joint_DA": float(row["joint_DA"]), "joint_CoT": float(row["joint_CoT"]),
                 "joint_gap": float(row["joint_gap"])}
                for k, row in joints[cls].iterrows()
            ] if len(joints[cls]) else []
            for cls in ladders
        },
        "not_selected": {cls: [c for c in chosen_all[cls] if False] for cls in chosen_all},
        "skipped_by_conflict": skipped_all,
        "rules": {c: rule_text(c) for c in CANDIDATES},
    }
    with open(OUT_LADDER, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_json_default)

    for cls in ("cond", "da_free"):
        print("\n" + "=" * 80)
        print("[%s] k=1..%s 阶梯" % (cls, len(ladders[cls])))
        for k, ids in enumerate(ladders[cls], 1):
            print("  k=%d: %s" % (k, " + ".join(ids)))
        if skipped_all[cls]:
            print("  因冲突跳过:", [(s["cons_id"], s["blocked_by"]) for s in skipped_all[cls]])
        if len(joints[cls]):
            display(joints[cls])
        else:
            print("  [警告] 该档没有可行约束。")

    print("\n输出文件:")
    for p in (OUT_RAW, OUT_IO, OUT_SUMMARY, OUT_VERDICTS, OUT_INDEP_F0, OUT_INDEP_CROSS, OUT_LADDER):
        print("  ", p)
    return summ, verdict, indep_f0, indep_cross, chosen_all, ladders, joints, payload


summ, verdict, indep_f0, indep_cross, CHOSEN, LADDERS, JOINTS, LADDER_PAYLOAD = finalize(cal)


## 13. 冲突矩阵与阶梯自检

这一段做三件事，都是纯离线检查：

1. 打出 16×16 冲突矩阵，确认互斥的只有那 3 对包裹型。
2. 断言阶梯的每条前缀都**无冲突**、且**嵌套**（k 是 k+1 的前缀）。
3. 用实际数据算一遍：如果把阶梯里的 5 条同时给模型，按单条通过率乘积估的 joint 是多少。

> 第 3 项是独立性假设下的估计。如果几条约束共享同一个失败原因（比如都要求数行），真实 joint 会高于乘积，
> 所以这个数对 joint 是**偏悲观**的——如果偏悲观的估计都有 gap，那真实数据大概率也有。


In [ ]:
conf_mat = conflict_matrix(ALL_CANDIDATES)
print("冲突矩阵（X = 不可共存）:")
display(conf_mat)

print("\n冲突对与理由:")
for item in conflict_list():
    print("  - %-10s x %-10s : %s" % (item["a"], item["b"], item["reason"]))
assert len(conflict_list()) == 0, "v11 的池子没有逻辑互斥对"

print("\n两条阶梯的自检:")
for cls in ("cond", "da_free"):
    ids_list = [list(x) for x in LADDERS[cls]]
    print("\n[%s] %d 级" % (cls, len(ids_list)))
    for k, ids in enumerate(ids_list, 1):
        assert is_compatible(ids), "k=%d 的约束集有冲突: %s" % (k, ids)
        assert len(ids) == k, (k, ids)
    for k in range(1, len(ids_list)):
        assert ids_list[k][:k] == ids_list[k - 1], "阶梯不是嵌套的"
    if ids_list:
        print("  自检通过：每级无冲突、且 k 是 k+1 的前缀")

print("\n每一级的规则全文（进正式实验前逐条读一遍）:")
for cls in ("cond", "da_free"):
    for k, ids in enumerate(LADDERS[cls], 1):
        print("=" * 90)
        print("[%s] k=%d" % (cls, k))
        for c in ids:
            print("  [%s | %s] %s" % (c, CONS[c]["cls"], rule_text(c)))


## 14. 用最终阶梯跑 k=1..5（正式剂量）

单约束校准只告诉你「每条单独能不能过」。真正的 k 轴要看**联合**：

- 这里直接拿第 12 节选出的阶梯，按前缀跑 `k=1`、`k=1..2`、…、`k=1..5`。
- 5 档 × 50 题 × 2 模式 = **500 条**。`k=1` 是一阶，`k=5` 是满配。
- 判分里 `strict_cons_ok` 已经是「本条件所有约束的 AND」，所以表格里的 `cons_pass` 就是 joint。

这张表才是「k 有没有剂量效应」的直接证据；前面 `JOINT` 那格只是按单条通过率乘出来的预测。


In [ ]:
if not RUN_LADDER:
    print("RUN_LADDER=False，跳过。")
else:
    all_lists = []
    for cls in ("cond", "da_free"):
        for ids in LADDERS[cls]:
            if ids not in all_lists:
                all_lists.append(ids)
    if not all_lists:
        print("两条阶梯都是空的，跳过。")
    else:
        lad_jobs = make_jobs("ladder", QUESTIONS, MODES, all_lists)
        print("阶梯 jobs:", len(lad_jobs), "| 档数:", len(all_lists))
        run_jobs(lad_jobs, path=OUT_LADDER_RAW, io_path=OUT_IO)

        lad = load_results(path=OUT_LADDER_RAW, stage="ladder")
        lad["cls"] = lad["cons_ids"].map(lambda ids: CONS[ids[0]]["cls"] if len(ids) else "?")
        display(lad.groupby(["cls", "k", "mode"]).agg(
            n=("id", "size"),
            joint=("strict_cons_ok", "mean"),
            base=("base_ok", "mean"),
            place=("placement_ok", "mean"),
            kw_after=("kw_after_answer", "mean"),
            answer=("answer_ok", "mean"),
            n_lines=("n_lines", "median"),
        ).round(3))

        print("\n两档 k 曲线（n=50/格）:")
        for cls in ("cond", "da_free"):
            sub = lad[lad["cls"] == cls]
            if not len(sub):
                continue
            j = sub.groupby(["k", "mode"])["strict_cons_ok"].mean().unstack("mode").reindex(columns=["DA", "CoT"])
            j["gap"] = j["DA"] - j["CoT"]
            print("\n[%s]" % cls)
            display(j.round(3))
            if len(j):
                p = j["DA"].dropna()
                print("  joint_DA: %s" % " -> ".join("%.2f" % v for v in p.values))


## 15. 人工审查清单（必须做）

- [ ] **§8 的基准率表**：6 条条件式的 `p_if_ignore` 是否都在 0.74–0.84。
- [ ] §11 里 `cond` 档的实测 `pass_DA` 离 `p_if_ignore` 有多远——远了说明模型真的在读条件。
- [ ] `A_*` 的条件用的是**模型自己写下的答案**（不是 gold），看输出里那个字母和行是否自洽。
- [ ] `kw_after_max` 是否仍然全 0。
- [ ] 两条 k 曲线：`cond` 是否按连乘下降、`da_free` 是否恒为 1。
- [ ] 如果 `cond` 的实测 p 远高于 `p_if_ignore`，说明模型几乎总能判断对条件；那 k 效应会很弱，
      要靠更多/更细的条件集合来加剂量。


In [ ]:
# 逐条打印 2 个样本，DA/CoT 都看
for c in ["F0"] + CANDIDATES:
    for mode in MODES:
        sub = cal[(cal["cons_id"] == c) & (cal["mode"] == mode)]
        if sub.empty:
            continue
        print("=" * 90)
        print("%s | %s | n=%s | strict=%.3f | base=%.3f | place=%.3f | answer=%.3f | da_extra=%.3f | n_lines=%s" % (
            c, mode, len(sub), sub["strict_cons_ok"].mean(), sub["base_ok"].mean(),
            sub["placement_ok"].mean(), sub["answer_ok"].mean(),
            sub["da_extra"].mean(), sub["n_lines"].median()))
        for _, r in sub.sample(min(2, len(sub)), random_state=SEED).iterrows():
            print("-" * 60)
            print(r["text"][:1200])

print("\n按 d 的 cons_pass:")
display(cal.pivot_table(index=["cons_id", "mode"], columns="d", values="strict_cons_ok").round(2))

print("\nF0 自然违反率（禁令类看 CoT，要求类看 DA）:")
f0_rows = []
for c in ALL_CANDIDATES:
    row = {"cons_id": c, "track": CONS[c]["track"], "kind": CONS[c]["kind"]}
    for mode in MODES:
        sub = cal[(cal["cons_id"] == "F0") & (cal["mode"] == mode)]
        row["F0_%s_pass" % mode] = sub["per_cons_all"].map(lambda d: _pass_all(d, c)).mean()
    f0_rows.append(row)
display(pd.DataFrame(f0_rows).set_index("cons_id").round(3))

print("\nI/O 日志前 2 行（解析示例）:")
if os.path.exists(OUT_IO):
    for i, line in enumerate(open(OUT_IO, encoding="utf-8")):
        if i >= 2:
            break
        r = json.loads(line)
        print("---", r["id"], "|", r["mode"], "|", r["cons_id"], "|", "ok=", r["strict_cons_ok"])
        print(r["output"][:300])


## 16. 下一步

1. 先看 §8 的基准率表（零成本），确认 6 个条件的 `1 - base` 都落在 0.74–0.84。
2. 看 §14 两条 k 曲线：
   - `cond` 的 `joint_DA` 按 `Π p` 下降 → **k 效应 = 独立条件失败率的连乘**，被显式复现；
   - `da_free` 恒为 1 → 对照组成立。
3. 如果 `cond` 的实测 p 明显高于 `p_if_ignore`（比如 0.95），说明模型几乎总能判对条件，
   那就把条件集合切得更细（更多互不重叠的小集合），用「更多条」而不是「更难」来加剂量。
4. 结果回填 `record.md`（v10 的结论也要补进去）。
